# 06 - Gold Tables
Reads the silver tables from notebooks 04 and 05 and writes the tables the dashboard uses.

| Gold table | Content |
|---|---|
| `gold_headline_kpis` | one row: gross, cannibalized and net units, confidence intervals, status |
| `gold_article_incremental_demand` | one row per discounted article (before/after lift and adjusted lift) |
| `gold_cannibalization_leakage` | one row per discounted article + exposed article |
| `gold_category_rollup` | gross, cannibalized and net units per product group + garment group |
| `gold_parallel_trends_demand` | copy of the silver table, for the chart |
| `gold_parallel_trends_cannibalization` | copy of the silver table, for the chart |
| `gold_pipeline_health` | one row added per run (history is kept) |

In [0]:
import math
import numpy as np
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

run_time = datetime.now()

In [0]:
# Settings
dbutils.widgets.text("articles_table", "workspace.default.hnm_articles_bronze")
dbutils.widgets.text("embeddings_schema", "workspace.hnm_discount")        # written by notebook 03
dbutils.widgets.text("silver_schema", "workspace.hnm_discount_silver")     # written by notebooks 04 and 05
dbutils.widgets.text("output_schema", "workspace.hnm_discount_gold")
dbutils.widgets.text("high_cannibalization_cutoff", "0.5")

articles_table = dbutils.widgets.get("articles_table")
embeddings_schema = dbutils.widgets.get("embeddings_schema")
silver_schema = dbutils.widgets.get("silver_schema")
output_schema = dbutils.widgets.get("output_schema")
high_cannibalization_cutoff = float(dbutils.widgets.get("high_cannibalization_cutoff"))

pre_gap_limit = 0.15        # parallel-trends warning level

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {output_schema}")

def save_table(sdf, name, mode="overwrite"):
    sdf.write.format("delta").mode(mode).option("overwriteSchema" if mode == "overwrite" else "mergeSchema", "true").saveAsTable(name)
    print("wrote", name)

In [0]:
# Load silver tables
article_units = spark.table(f"{silver_schema}.hnm_article_treatment_weekly")
pair_effects = spark.table(f"{silver_schema}.hnm_cannibalization_pair_effects")
inc = spark.table(f"{silver_schema}.hnm_incremental_demand_did").toPandas().iloc[0]
can = spark.table(f"{silver_schema}.hnm_cannibalization_did").toPandas().iloc[0]

article_names = (spark.table(articles_table)
                 .select(F.col("article_id").cast("string").alias("article_id"), "prod_name")
                 .dropDuplicates(["article_id"]))

In [0]:
# Article level: one row per discounted article
# lift_units = week 5 minus the 'before' average (NOT causal), adj_lift_units = lift minus the controls' change (DiD)
gold_article = (article_units
                .filter((F.col("T") == 1) & (F.col("is_matched") == 1))
                .select("article_id", "prod_name", "price", "discount_pct", "baseline_units", "week5_units",
                        "lift_units", "control_delta", "adj_lift_units", "colour_group_name", "garment_group_name",
                        "department_name", "product_group_name", "index_name", "product_type_name", "section_name")
                .withColumn("run_ts", F.lit(run_time)))
save_table(gold_article, f"{output_schema}.gold_article_incremental_demand")

wrote workspace.hnm_discount_gold.gold_article_incremental_demand


In [0]:
# Cannibalization level: one row per discounted article + exposed article
gold_cannibalization = (pair_effects
                        .select(F.col("anchor_id").alias("anchor_article_id"),
                                F.col("candidate_id").alias("substitute_article_id"),
                                F.col("delta").alias("substitute_unit_change"),
                                F.col("adj_change").alias("substitute_adj_unit_change"),
                                "final_score")
                        .join(article_names.select(F.col("article_id").alias("anchor_article_id"),
                                                   F.col("prod_name").alias("anchor_prod_name")),
                              on="anchor_article_id", how="left")
                        .join(article_names.select(F.col("article_id").alias("substitute_article_id"),
                                                   F.col("prod_name").alias("substitute_prod_name")),
                              on="substitute_article_id", how="left")
                        .withColumn("run_ts", F.lit(run_time)))
save_table(gold_cannibalization, f"{output_schema}.gold_cannibalization_leakage")

wrote workspace.hnm_discount_gold.gold_cannibalization_leakage


In [0]:
# Category rollup (category of the discounted article)
category_cols = ["product_group_name", "garment_group_name"]

category_lift = gold_article.groupBy(*category_cols).agg(
    F.sum("adj_lift_units").alias("gross_incremental_units"),
    F.count("article_id").alias("n_discounted_articles"))

anchor_categories = article_units.select(F.col("article_id").alias("anchor_article_id"), *category_cols)

# cannibalized units = minus the change in the exposed articles' sales (positive = lost sales)
category_cannibalized = (gold_cannibalization
                         .join(anchor_categories, on="anchor_article_id", how="inner")
                         .groupBy(*category_cols)
                         .agg((-F.sum("substitute_adj_unit_change")).alias("cannibalized_units")))

gold_rollup = (category_lift.join(category_cannibalized, on=category_cols, how="outer")
               .fillna(0, subset=["gross_incremental_units", "n_discounted_articles", "cannibalized_units"])
               .withColumn("net_incremental_units", F.col("gross_incremental_units") - F.col("cannibalized_units"))
               .withColumn("pct_lift_cannibalized",
                           F.when(F.col("gross_incremental_units") > 0,
                                  F.col("cannibalized_units") / F.col("gross_incremental_units")))
               .withColumn("high_cannibalization_flag",
                           F.coalesce(F.col("pct_lift_cannibalized") > high_cannibalization_cutoff, F.lit(False)))
               .withColumn("run_ts", F.lit(run_time)))
save_table(gold_rollup, f"{output_schema}.gold_category_rollup")

wrote workspace.hnm_discount_gold.gold_category_rollup


In [0]:
# Headline KPIs
n_treated = float(inc["n_treated"])
n_subs = float(can["n_exposed_substitutes"])

gross_units = float(inc["did_estimate"]) * n_treated          # effect per discounted article x number of them
cannibalized_units = float(can["cannibalized_units_total"])   # each exposed article counted once
net_units = gross_units - cannibalized_units

# confidence interval of net units (treats the two estimates as independent, a simplification)
se_gross = (inc["did_ci_high"] - inc["did_ci_low"]) / (2 * 1.96) * n_treated
se_cann = (can["cannibalization_ci_high"] - can["cannibalization_ci_low"]) / (2 * 1.96) * n_subs
se_net = math.sqrt(se_gross ** 2 + se_cann ** 2)
net_low, net_high = net_units - 1.96 * se_net, net_units + 1.96 * se_net

# the category table must add up to the headline numbers
totals = gold_rollup.agg(F.sum("gross_incremental_units").alias("g"), F.sum("cannibalized_units").alias("c")).first()
assert abs(totals["g"] - gross_units) <= 1e-6 * max(1.0, abs(gross_units)), "category gross does not add up"
assert abs(totals["c"] - cannibalized_units) <= 1e-6 * max(1.0, abs(cannibalized_units)), "category cannibalized does not add up"

# share cannibalized only makes sense when gross is positive
pct_lift_cannibalized = cannibalized_units / gross_units if gross_units > 0 else np.nan

# status text for a dashboard banner
problems = []
if gross_units <= 0:
    problems.append("NO_POSITIVE_GROSS_EFFECT")
if max(float(inc["pre_gap_max"]), float(can["pre_gap_max"])) > pre_gap_limit:
    problems.append("PRE_TREND_WARNING")
kpi_status = "OK" if len(problems) == 0 else ";".join(problems)

kpis = pd.DataFrame([{
    "run_ts": run_time,
    "incremental_demand_ate": float(inc["did_estimate"]),
    "incremental_demand_ci_low": float(inc["did_ci_low"]),
    "incremental_demand_ci_high": float(inc["did_ci_high"]),
    "n_treated": n_treated,
    "n_control": float(inc["n_control"]),
    "cannibalization_ate": float(can["cannibalization_ate"]),
    "cannibalization_ci_low": float(can["cannibalization_ci_low"]),
    "cannibalization_ci_high": float(can["cannibalization_ci_high"]),
    "cannibalization_p_value": float(can["cannibalization_pvalue"]),
    "n_treated_pairs": float(can["n_pairs"]),
    "n_exposed_substitutes": n_subs,
    "gross_incremental_units": gross_units,
    "cannibalized_units": cannibalized_units,
    "net_incremental_units": net_units,
    "net_incremental_units_ci_low": net_low,
    "net_incremental_units_ci_high": net_high,
    "pct_lift_cannibalized": pct_lift_cannibalized,
    "incremental_demand_ci_range": f"[{inc['did_ci_low']:.2f}, {inc['did_ci_high']:.2f}]",
    "cannibalization_ci_range": f"[{can['cannibalization_ci_low']:.2f}, {can['cannibalization_ci_high']:.2f}]",
    "cannibalization_ate_ci": f"[{can['cannibalization_ci_low']:.2f}, {can['cannibalization_ci_high']:.2f}], p={can['cannibalization_pvalue']:.4f}",
    "net_incremental_units_ci_range": f"[{net_low:.2f}, {net_high:.2f}]",
    "incremental_pre_gap_max": float(inc["pre_gap_max"]),
    "cannibalization_pre_gap_max": float(can["pre_gap_max"]),
    "kpi_status": kpi_status,
}])

gold_kpis = (spark.createDataFrame(kpis)
             .withColumn("pct_lift_cannibalized",           # NaN -> NULL so the dashboard shows blank
                         F.when(F.isnan("pct_lift_cannibalized"), F.lit(None).cast("double"))
                          .otherwise(F.col("pct_lift_cannibalized"))))
save_table(gold_kpis, f"{output_schema}.gold_headline_kpis")
print("kpi_status:", kpi_status)
display(gold_kpis)

wrote workspace.hnm_discount_gold.gold_headline_kpis
kpi_status: NO_POSITIVE_GROSS_EFFECT


run_ts,incremental_demand_ate,incremental_demand_ci_low,incremental_demand_ci_high,n_treated,n_control,cannibalization_ate,cannibalization_ci_low,cannibalization_ci_high,cannibalization_p_value,n_treated_pairs,n_exposed_substitutes,gross_incremental_units,cannibalized_units,net_incremental_units,net_incremental_units_ci_low,net_incremental_units_ci_high,pct_lift_cannibalized,incremental_demand_ci_range,cannibalization_ci_range,cannibalization_ate_ci,net_incremental_units_ci_range,incremental_pre_gap_max,cannibalization_pre_gap_max,kpi_status
2026-10-01T19:46:39.653Z,-4.682931220741883,-5.900655126915728,-3.4652073145680387,2777.0,4440.0,-3.7592480854030526,-5.111178910625209,-2.4073172601808963,5.037655388706551E-8,7946.0,4309.0,-13004.500000000211,16198.6,-29203.10000000021,-35938.93319735429,-22467.266802646132,null,"[-5.90, -3.47]","[-5.11, -2.41]","[-5.11, -2.41], p=0.0000","[-35938.93, -22467.27]",0.022970465544555674,0.023151430723000466,NO_POSITIVE_GROSS_EFFECT


In [0]:
# Parallel-trends tables for the dashboard charts (copied from silver so the dashboard only reads gold)
save_table(spark.table(f"{silver_schema}.parallel_trends_demand"), f"{output_schema}.gold_parallel_trends_demand")
save_table(spark.table(f"{silver_schema}.parallel_trends_cannibalization"), f"{output_schema}.gold_parallel_trends_cannibalization")

wrote workspace.hnm_discount_gold.gold_parallel_trends_demand
wrote workspace.hnm_discount_gold.gold_parallel_trends_cannibalization


In [0]:
# Pipeline health: one row per run (append, so we keep the history)
meta = spark.table(f"{embeddings_schema}.hnm_embedding_run_meta").orderBy(F.col("run_ts").desc()).first()
embeddings_run_ts = meta["run_ts"]

health = pd.DataFrame([{
    "run_ts": run_time,
    "n_articles_analyzed": int(article_units.count()),
    "n_discounted": int(article_units.filter(F.col("T") == 1).count()),
    "n_non_discounted": int(article_units.filter(F.col("T") == 0).count()),
    "n_neighbor_rows": int(spark.table(f"{embeddings_schema}.hnm_article_text_nn_candidates").count()),
    "incremental_n_treated": int(inc["n_treated"]),
    "incremental_n_control": int(inc["n_control"]),
    "incremental_n_control_pool": int(inc["n_control_pool"]),
    "incremental_r_squared": float(inc["r_squared"]),
    "n_exposed_substitutes": int(n_subs),
    "embeddings_last_run_ts": embeddings_run_ts,
    "embedding_age_hours": (run_time - embeddings_run_ts).total_seconds() / 3600,
    "pre_gap_max_incremental": float(inc["pre_gap_max"]),
    "pre_gap_max_cannibalization": float(can["pre_gap_max"]),
    "kpi_status": kpi_status,
}])

save_table(spark.createDataFrame(health), f"{output_schema}.gold_pipeline_health", mode="append")
display(spark.table(f"{output_schema}.gold_pipeline_health").orderBy(F.col("run_ts").desc()).limit(5))

wrote workspace.hnm_discount_gold.gold_pipeline_health


run_ts,n_articles_analyzed,n_discounted,n_non_discounted,n_neighbor_rows,incremental_n_treated,incremental_n_control,incremental_n_control_pool,incremental_r_squared,n_exposed_substitutes,embeddings_last_run_ts,embedding_age_hours,pre_gap_max_incremental,pre_gap_max_cannibalization,kpi_status
2026-10-01T19:46:39.653Z,14519,2777,11742,3166260,2777,4440,7433,0.024760075959443073,4309,2026-10-01T19:37:18.021Z,0.15600883888888886,0.022970465544555674,0.023151430723000466,NO_POSITIVE_GROSS_EFFECT
2026-10-01T02:54:24.752Z,14519,2777,11742,3166260,2777,4440,7433,0.024760075959443073,4309,2026-10-01T00:31:16.861Z,2.3855252383333334,0.022970465544555674,0.023151430723000466,NO_POSITIVE_GROSS_EFFECT
